# Qwen3 speculative decoding on Kaggle T4 x2
Run top-to-bottom in a fresh **GPU T4 x2** session. Every command is fail-fast; do not continue after a failed gate.

In [ ]:
import json, os, shlex, subprocess
from datetime import datetime, timezone
from pathlib import Path
RUN = Path('results/kaggle-speculative-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
RUN.mkdir(parents=True, exist_ok=False)
os.environ['RUN'] = str(RUN.resolve())
def run(command):
    print('+', command)
    subprocess.run(command, shell=True, check=True, env=os.environ.copy())
print(RUN)

In [ ]:
run('bash scripts/setup_kaggle.sh')
run('python scripts/kaggle_preflight.py --require-t4-count 2 --out "$RUN/preflight.json"')
run('git rev-parse HEAD > "$RUN/git_sha.txt"')
run('git status --short > "$RUN/git_status.txt"')
run('nvidia-smi topo -m > "$RUN/nvidia_topology.txt"')

In [ ]:
run('python -m pytest -q -m "not cuda" -p no:cacheprovider')
run('python -m pytest -q -m cuda -p no:cacheprovider tests/speculative tests/batching tests/cache')

In [ ]:
for draft, label in [('Qwen/Qwen3-0.6B', '06b'), ('Qwen/Qwen3-1.7B', '17b')]:
    run(f'python -m benchmarks.speculative.pair_screen --draft-model {shlex.quote(draft)} --depths 2,3,4 --warmup 1 --runs 3 --max-new-tokens 64 --output "$RUN/pair_{label}.json"')
run('python -m benchmarks.speculative.select_pair "$RUN/pair_06b.json" "$RUN/pair_17b.json" --output "$RUN/pair_selection.json"')
selection = json.loads((RUN / 'pair_selection.json').read_text())
selection['winner']

In [ ]:
winner = selection['winner']
draft = shlex.quote(winner['draft_model'])
depth = int(winner['depth'])
run(f'python -m benchmarks.speculative.engine_ab --draft-model {draft} --depth {depth} --concurrencies 1,2,4 --num-blocks 512 --warmup 2 --runs 5 --max-new-tokens 128 --output "$RUN/engine_ab.json"')
json.loads((RUN / 'engine_ab.json').read_text())['rows']

The pair is accepted only if both screening and the live paged-engine A/B exit successfully. The checked-in server factory uses the expected 4B/0.6B/depth-3 profile; if the measured winner differs, pass the selected model and depth to `create_app` rather than using that factory. Preserve the entire run directory.